# EduPath Baseline Academic Performance Model
This notebook explores the UCI Student Performance dataset to predict final grades (G3). This model will serve as a baseline academic predictor, separate from the career recommendation engine.

In [ ]:
import pandas as pd

# Note: The UCI student dataset uses semicolons as separators
df = pd.read_csv("../../data/raw/uci_student_performance/student-mat.csv", sep=";")

print(df.head())
print("Shape:", df.shape)
print("\nColumns:", df.columns)
print("\nInfo:")
df.info()

In [ ]:
print(df.describe())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicated rows:\n", df.duplicated().sum())

### Target Variable and Data Leakage Prevention
We will predict `G3` (final grade). We MUST drop `G1` and `G2` to prevent data leakage, simulating a prediction made before these intermediate grades are known.

In [ ]:
# Define features and target
X = df.drop(columns=["G1", "G2", "G3"])
y = df["G3"]

categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

print("Categorical Columns:", categorical_cols.tolist())
print("Numeric Columns:", numeric_cols.tolist())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols.tolist()),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols.tolist())
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Model Training and Evaluation
We will test Linear Regression, Random Forest, and Gradient Boosting.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, random_state=42)
}

results = []
best_model = None
best_r2 = -float('inf')

for name, model_cls in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model_cls)
    ])
    
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    
    if r2 > best_r2:
        best_r2 = r2
        best_model = pipeline

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

### Save the Model

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/student_performance_model.pkl")
print("Model saved successfully to ml/models/student_performance_model.pkl")